---
## ⚙️ SETUP — Wajib Dijalankan Pertama Kali
> Jalankan semua cell setup di bawah sebelum memulai latihan SQL.
---

In [1]:
# LANGKAH 1: Autentikasi Google Cloud
from google.colab import auth
auth.authenticate_user()
print('Autentikasi berhasil!')

Autentikasi berhasil!


In [2]:
# LANGKAH 2: Set Project ID & BigQuery Client
from google.cloud import bigquery

project_id = 'data-project-agus'  # <- GANTI dengan Project ID Google Cloud Anda

client = bigquery.Client(project=project_id)
print(f'BigQuery client siap. Project: {project_id}')

BigQuery client siap. Project: data-project-agus


### Langkah 3: Upload CSV ke BigQuery
> **Sebelum menjalankan cell ini:**
> 1. Klik ikon folder di sidebar kiri Colab
> 2. Upload semua file CSV berikut:
>    - `customer_demography.csv`
>    - `customer_location.csv`
>    - `customer_status.csv`
>    - `population.csv`
>    - `telco_services.csv`

In [3]:
# LANGKAH 3: Upload CSV ke BigQuery sebagai tabel
import pandas as pd

dataset_name = 'telco_churn'

# Buat dataset jika belum ada
dataset_ref = bigquery.Dataset(f'{project_id}.{dataset_name}')
dataset_ref.location = 'US'
try:
    client.create_dataset(dataset_ref)
    print(f"Dataset '{dataset_name}' berhasil dibuat!")
except Exception:
    print(f"Dataset '{dataset_name}' sudah ada.")

# Upload setiap tabel dari CSV
csv_files = {
    'customer_demography': 'customer_demography.csv',
    'customer_location'  : 'customer_location.csv',
    'customer_status'    : 'customer_status.csv',
    'population'         : 'population.csv',
    'telco_services'     : 'telco_services.csv',
}

for table_name, csv_file in csv_files.items():
    df = pd.read_csv(csv_file)
    table_id = f'{project_id}.{dataset_name}.{table_name}'
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    tbl = client.get_table(table_id)
    print(f'{table_name}: {tbl.num_rows} baris berhasil diupload')

print('Semua tabel berhasil diupload ke BigQuery!')

Dataset 'telco_churn' berhasil dibuat!
customer_demography: 7043 baris berhasil diupload
customer_location: 7043 baris berhasil diupload
customer_status: 7043 baris berhasil diupload
population: 1671 baris berhasil diupload
telco_services: 7043 baris berhasil diupload
Semua tabel berhasil diupload ke BigQuery!


---
## Mulai Latihan SQL
---

<a id="A"></a>
# <b>A. <span style='color:#0B2F9F'><code>SELECT</code></span></b>

Perintah `SELECT` bertujuan untuk mengambil dan menampilkan data dari tabel di database. Ini penting untuk analisis karena saat menganalisa data perlu mengakses data yang relevan untuk dianalisis seperti mengidentifikasi pola, tren, dan informasi penting lainnya.

<a id="A.1."></a>
## <b>A.1. <span style='color:#0B2F9F'>Menampilkan Semua Data pada Satu Tabel</span></b>

Untuk menampilkan semua isi data pada sebuah tabel, gunakan tanda bintang (`*`).

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-Select1.png' width='50%'>

*docs : <a href='https://dev.mysql.com/doc/refman/8.4/en/select.html'>https://dev.mysql.com/doc/refman/8.4/en/select.html</a>*

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan semua data pada tabel customer_status!</b>*

In [4]:
# A.1 - Tampilkan semua data customer_status
sql = f"""
SELECT *
FROM `{project_id}.telco_churn.customer_status`
"""
client.query(sql).to_dataframe()

,customer_id,satisfaction_score,status,churn_label,churn_score,cltv,churn_category,churn_reason
0,1989-PRJHP,1,Churned,Yes,65,2377,Competitor,Competitor had better devices
1,7216-EWTRS,1,Churned,Yes,65,3128,Attitude,Attitude of service provider
2,3269-ATYWD,1,Churned,Yes,65,4892,Other,Don't know
3,8510-AWCXC,1,Churned,Yes,65,3269,Attitude,Attitude of support person
4,9907-SWKKF,1,Churned,Yes,65,4842,Competitor,Competitor had better devices
...,...,...,...,...,...,...,...,...
7038,0856-NAOES,5,Stayed,No,80,2191,None,None
7039,6549-BTYPG,5,Stayed,No,80,5784,None,None
7040,5146-YYFRZ,5,Stayed,No,80,6212,None,None
7041,5351-QESIO,5,Joined,No,80,4276,None,None


<a id="A.2."></a>
## <b>A.2. <span style='color:#0B2F9F'>Menampilkan Kolom Tertentu pada Satu Tabel</span></b>

Untuk dapat menampilkan isi data pada kolom tertentu pada sebuah tabel, eksekusi perintah

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-Select2.png' width='50%'>

Gunakan tanda koma untuk mendaftarkan nama kolom yang ingin ditampilkan.

*docs : <a href='https://dev.mysql.com/doc/refman/8.4/en/selecting-columns.html'>https://dev.mysql.com/doc/refman/8.4/en/selecting-columns.html</a>*

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan kolom customer_id, status dan churn_reason pada tabel customer_status!</b>*

In [5]:
# A.2 - Tampilkan kolom customer_id, status, churn_reason
sql = f"""
SELECT customer_id, status, churn_reason
FROM `{project_id}.telco_churn.customer_status`
"""
client.query(sql).to_dataframe()

,customer_id,status,churn_reason
0,1989-PRJHP,Churned,Competitor had better devices
1,7216-EWTRS,Churned,Attitude of service provider
2,3269-ATYWD,Churned,Don't know
3,8510-AWCXC,Churned,Attitude of support person
4,9907-SWKKF,Churned,Competitor had better devices
...,...,...,...
7038,0856-NAOES,Stayed,None
7039,6549-BTYPG,Stayed,None
7040,5146-YYFRZ,Stayed,None
7041,5351-QESIO,Joined,None


<a id="A.3."></a>
## <b>A.3. <span style='color:#0B2F9F'>Mengubah Penyebutan Nama Kolom pada Satu Tabel</span></b>

Untuk mengubah nama sementara kolom dalam query bisa menggunakan perintah `AS` (Alias).

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-Select3.png' width='50%'>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan kolom customer_id, cltv dan status pada tabel customer_status, namun ubah sementara nama kolom cltv menjadi customer_lifetime_value agar lebih mudah diinterpretasikan!</b>*

In [6]:
# A.3 - Tampilkan dengan alias kolom cltv -> customer_lifetime_value
sql = f"""
SELECT customer_id, cltv AS customer_lifetime_value, status
FROM `{project_id}.telco_churn.customer_status`
"""
client.query(sql).to_dataframe()

,customer_id,customer_lifetime_value,status
0,1989-PRJHP,2377,Churned
1,7216-EWTRS,3128,Churned
2,3269-ATYWD,4892,Churned
3,8510-AWCXC,3269,Churned
4,9907-SWKKF,4842,Churned
...,...,...,...
7038,0856-NAOES,2191,Stayed
7039,6549-BTYPG,5784,Stayed
7040,5146-YYFRZ,6212,Stayed
7041,5351-QESIO,4276,Joined


<a id="B"></a>
# <b>B. <span style='color:#E1B12D'><code>DISTINCT</code></span></b>

Perintah `DISTINCT` digunakan untuk menghapus duplikasi dalam hasil query.

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-Distinct.png' width='50%'>

docs : <a href='https://dev.mysql.com/doc/refman/8.4/en/distinct-optimization.html'>*https://dev.mysql.com/doc/refman/8.4/en/distinct-optimization.html*</a>

<a id="B.1."></a>
## <b>B.1. <span style='color:#0B2F9F'>Nilai Unik pada Satu Kolom Tertentu</span></b>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan apa saja jenis status pada tabel customer_status!</b>*

In [7]:
# B.1 - Tampilkan nilai unik kolom status
sql = f"""
SELECT DISTINCT status
FROM `{project_id}.telco_churn.customer_status`
"""
client.query(sql).to_dataframe()

,status
0,Churned
1,Stayed
2,Joined


<a id="B.2."></a>
## <b>B.2. <span style='color:#0B2F9F'>Nilai Unik pada Lebih dari Satu Kolom</span></b>

Perintah `DISTINCT` dapat diterapkan di lebih dari satu kolom. Jika diterapkan pada lebih dari satu kolom maka akan menghasilkan kombinasi unik antar baris.

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan apa saja jenis churn_category dan churn_reason pada tabel customer_status!</b>*

In [8]:
# B.2 - Tampilkan kombinasi unik churn_category & churn_reason
sql = f"""
SELECT DISTINCT churn_category, churn_reason
FROM `{project_id}.telco_churn.customer_status`
"""
client.query(sql).to_dataframe()

,churn_category,churn_reason
0,Competitor,Competitor had better devices
1,Attitude,Attitude of service provider
2,Other,Don't know
3,Attitude,Attitude of support person
4,Price,Price too high
5,Dissatisfaction,Product dissatisfaction
6,Competitor,Competitor made better offer
7,Dissatisfaction,Service dissatisfaction
8,Competitor,Competitor offered higher download speeds
9,Competitor,Competitor offered more data


<a id="C"></a>
# <b>C. <span style='color:#E1B12D'><code>LIMIT</code></span></b>

Perintah `LIMIT <jumlah_data>` digunakan untuk membatasi data yang ingin ditampilkan. `LIMIT` diletakkan paling akhir dari sebuah query.

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-Limit.png' width='50%'>

docs : <a href='https://dev.mysql.com/doc/refman/8.4/en/limit-optimization.html'>*https://dev.mysql.com/doc/refman/8.4/en/limit-optimization.html*</a>

<a id="C.1."></a>
## <b>C.1. <span style='color:#0B2F9F'>Membatasi Jumlah Baris (yang ditampilkan) di Semua Kolom pada Data</span></b>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan semua 5 data teratas pada tabel customer_status!</b>*

In [9]:
# C.1 - Tampilkan 5 data teratas semua kolom
sql = f"""
SELECT *
FROM `{project_id}.telco_churn.customer_status`
LIMIT 5
"""
client.query(sql).to_dataframe()

,customer_id,satisfaction_score,status,churn_label,churn_score,cltv,churn_category,churn_reason
0,1989-PRJHP,1,Churned,Yes,65,2377,Competitor,Competitor had better devices
1,7216-EWTRS,1,Churned,Yes,65,3128,Attitude,Attitude of service provider
2,3269-ATYWD,1,Churned,Yes,65,4892,Other,Don't know
3,8510-AWCXC,1,Churned,Yes,65,3269,Attitude,Attitude of support person
4,9907-SWKKF,1,Churned,Yes,65,4842,Competitor,Competitor had better devices


<a id="C.2."></a>
## <b>C.2. <span style='color:#0B2F9F'>Membatasi Jumlah Baris (yang ditampilkan) di Kolom Tertentu pada Data</span></b>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan 10 data teratas untuk kolom customer_id, status dan churn_reason pada tabel customer_status!</b>*

In [10]:
# C.2 - Tampilkan 10 data teratas kolom tertentu
sql = f"""
SELECT customer_id, status, churn_reason
FROM `{project_id}.telco_churn.customer_status`
LIMIT 10
"""
client.query(sql).to_dataframe()

,customer_id,status,churn_reason
0,1989-PRJHP,Churned,Competitor had better devices
1,7216-EWTRS,Churned,Attitude of service provider
2,3269-ATYWD,Churned,Don't know
3,8510-AWCXC,Churned,Attitude of support person
4,9907-SWKKF,Churned,Competitor had better devices
5,2956-GGUCQ,Churned,Attitude of service provider
6,0447-BEMNG,Churned,Price too high
7,5949-XIKAE,Churned,Product dissatisfaction
8,0822-GAVAP,Churned,Competitor made better offer
9,3891-PUQOD,Churned,Service dissatisfaction


#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan 3 data teratas untuk kolom customer_id, cltv dan status pada tabel customer_status, namun ubah sementara nama kolom cltv menjadi customer_lifetime_value agar lebih mudah diinterpretasikan!</b>*

In [11]:
# C.2 (2) - Tampilkan 3 data teratas dengan alias cltv
sql = f"""
SELECT customer_id, cltv AS customer_lifetime_value, status
FROM `{project_id}.telco_churn.customer_status`
LIMIT 3
"""
client.query(sql).to_dataframe()

,customer_id,customer_lifetime_value,status
0,1989-PRJHP,2377,Churned
1,7216-EWTRS,3128,Churned
2,3269-ATYWD,4892,Churned


<a id="D"></a>
# <b>D. <span style='color:#E1B12D'><code>ORDER BY</code></span></b>

Perintah `ORDER BY` digunakan untuk mengurutkan hasil query berdasarkan satu atau lebih kolom.
<ul>
    <li><b>Ascending</b> : Pengurutan secara naik (dari kecil ke besar) atau A-Z</li>
    <li><b>Descending</b> : Pengurutan secara turun (dari besar ke kecil) atau Z-A</li>
</ul>

*docs : <a href='https://dev.mysql.com/doc/refman/8.4/en/sorting-rows.html'>https://dev.mysql.com/doc/refman/8.4/en/sorting-rows.html</a>*

<a id="D.1."></a>
## <b>D.1. <span style='color:#0B2F9F'>Mengurutkan secara <i>Ascending</i></span></b>

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-OrderByASC.png' width='50%'>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan semua data pada tabel customer_status namun urutkan berdasarkan kolom customer_id secara ascending!</b>*

In [12]:
# D.1 - Urutkan berdasarkan customer_id secara ASC
sql = f"""
SELECT *
FROM `{project_id}.telco_churn.customer_status`
ORDER BY customer_id ASC
"""
client.query(sql).to_dataframe()

,customer_id,satisfaction_score,status,churn_label,churn_score,cltv,churn_category,churn_reason
0,0002-ORFBO,3,Stayed,No,65,2205,None,None
1,0003-MKNFE,5,Stayed,No,66,5414,None,None
2,0004-TLHLJ,1,Churned,Yes,71,4479,Competitor,Competitor had better devices
3,0011-IGKFF,1,Churned,Yes,91,3714,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,1,Churned,Yes,68,3464,Dissatisfaction,Network reliability
...,...,...,...,...,...,...,...,...
7038,9987-LUTYD,4,Stayed,No,59,3161,None,None
7039,9992-RRAMN,1,Churned,Yes,68,5248,Dissatisfaction,Product dissatisfaction
7040,9992-UJOEL,5,Joined,No,33,5870,None,None
7041,9993-LHIEB,3,Stayed,No,59,4792,None,None


<a id="D.2."></a>
## <b>D.2. <span style='color:#0B2F9F'>Mengurutkan secara <i>Descending</i></span></b>

<img src='https://raw.githubusercontent.com/bachtiyarma/Material/main/Image/Materi-SQL/SQL-OrderByDESC.png' width='50%'>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan semua data pada tabel customer_status namun urutkan berdasarkan kolom cltv secara descending!</b>*

In [13]:
# D.2 - Urutkan berdasarkan cltv secara DESC
sql = f"""
SELECT *
FROM `{project_id}.telco_churn.customer_status`
ORDER BY cltv DESC
"""
client.query(sql).to_dataframe()

,customer_id,satisfaction_score,status,churn_label,churn_score,cltv,churn_category,churn_reason
0,7622-FWGEW,3,Stayed,No,70,6500,None,None
1,6024-RUGGH,3,Stayed,No,62,6499,None,None
2,0383-CLDDA,3,Stayed,No,37,6499,None,None
3,2683-JXWQQ,3,Stayed,No,64,6495,None,None
4,4114-QMKVN,3,Stayed,No,53,6494,None,None
...,...,...,...,...,...,...,...,...
7038,7928-VJYAB,5,Stayed,No,65,2004,None,None
7039,4657-FWVFY,5,Stayed,No,52,2004,None,None
7040,0871-URUWO,1,Churned,Yes,66,2003,Competitor,Competitor had better devices
7041,4925-LMHOK,2,Churned,Yes,90,2003,Dissatisfaction,Service dissatisfaction


<a id="D.3."></a>
## <b>D.3. <span style='color:#0B2F9F'>Kombinasi <i>Ascending</i> & <i>Descending</i> saat Mengurutkan Data</span></b>

#### *<b><span style='color:#55679C'>Quest</span> : Tampilkan 10 data pada tabel customer_status namun urutkan berdasarkan customer_id dari yang terkecil hingga yang terbesar dan urutkan pula cltv dari yang terbesar hingga yang terkecil!</b>*

In [14]:
# D.3 - Kombinasi ORDER BY ASC & DESC dengan LIMIT
sql = f"""
SELECT *
FROM `{project_id}.telco_churn.customer_status`
ORDER BY customer_id ASC, cltv DESC
LIMIT 10
"""
client.query(sql).to_dataframe()

,customer_id,satisfaction_score,status,churn_label,churn_score,cltv,churn_category,churn_reason
0,0002-ORFBO,3,Stayed,No,65,2205,None,None
1,0003-MKNFE,5,Stayed,No,66,5414,None,None
2,0004-TLHLJ,1,Churned,Yes,71,4479,Competitor,Competitor had better devices
3,0011-IGKFF,1,Churned,Yes,91,3714,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,1,Churned,Yes,68,3464,Dissatisfaction,Network reliability
5,0013-MHZWF,3,Stayed,No,55,5108,None,None
6,0013-SMEOE,3,Stayed,No,26,5011,None,None
7,0014-BMAQU,4,Stayed,No,49,4604,None,None
8,0015-UOCOJ,3,Stayed,No,34,5525,None,None
9,0016-QLJIS,3,Stayed,No,25,5509,None,None


In [15]:
# D.3 contoh lain - Top 10 CLTV dengan alias
sql = f"""
SELECT customer_id, cltv AS customer_lifetime_value, status
FROM `{project_id}.telco_churn.customer_status`
ORDER BY cltv DESC
LIMIT 10
"""
client.query(sql).to_dataframe()

,customer_id,customer_lifetime_value,status
0,7622-FWGEW,6500,Stayed
1,6024-RUGGH,6499,Stayed
2,0383-CLDDA,6499,Stayed
3,2683-JXWQQ,6495,Stayed
4,4114-QMKVN,6494,Stayed
5,8894-JVDCV,6494,Stayed
6,4531-AUZNK,6492,Stayed
7,2181-TIDSV,6492,Stayed
8,1658-XUHBX,6492,Stayed
9,0675-NCDYU,6491,Stayed


#### *<b><span style='color:#55679C'>Quest</span> : Siapa saja customer yang berada di top 10 nilai cltv terbesar? Tampilkan kolom customer_id, cltv dan status pada tabel customer_status, namun ubah sementara nama kolom cltv menjadi customer_lifetime_value agar lebih mudah diinterpretasikan</b>*

In [16]:
# D.4 - Siapa saja top 10 customer dengan CLTV terbesar?
sql = f"""
SELECT customer_id, cltv AS customer_lifetime_value, status
FROM `{project_id}.telco_churn.customer_status`
ORDER BY cltv DESC
LIMIT 10
"""
client.query(sql).to_dataframe()

,customer_id,customer_lifetime_value,status
0,7622-FWGEW,6500,Stayed
1,6024-RUGGH,6499,Stayed
2,0383-CLDDA,6499,Stayed
3,2683-JXWQQ,6495,Stayed
4,4114-QMKVN,6494,Stayed
5,8894-JVDCV,6494,Stayed
6,4531-AUZNK,6492,Stayed
7,2181-TIDSV,6492,Stayed
8,1658-XUHBX,6492,Stayed
9,0675-NCDYU,6491,Stayed


Data Source : *https://www.ibm.com/docs/en/cognos-analytics/11.1.0?topic=samples-telco-customer-churn*